# 08 — Giải thích mô hình bằng XAI và Cổng kiểm soát Gate D17

## 1. Mục tiêu
Notebook này giải thích và trực quan hóa **Bước 8** trong quy trình XAI-Football:
- Áp dụng các thuật toán Explainable AI (XAI) trên đồ thị để giải thích dự đoán của mô hình MPNN:
  1. **GNNExplainer:** Tối ưu hóa mặt nạ cạnh mềm và mặt nạ thuộc tính đỉnh dựa trên lý thuyết thông tin lẫn nhau (Mutual Information).
  2. **PGExplainer:** Sử dụng mạng nơ-ron tham số hóa (Parameterized Graph Explainer) học cách dự đoán xác suất giữ lại từng cạnh.
  3. **Random Baseline:** Explainer ngẫu nhiên gán trọng số ngẫu nhiên đều cho các cạnh để làm mốc đối chuẩn.
  4. **Last-edge Heuristic:** Heuristic gắn toàn bộ trọng số giải thích cho đường chuyền cuối cùng trước cú sút.
- Đánh giá định lượng trên **1.222 đồ thị** thuộc tập Test (57 trận đấu độc lập) thông qua các độ đo:
  - **Fidelity+:** Độ sụt giảm xác suất khi loại bỏ top-k cạnh quan trọng nhất.
  - **Fidelity-:** Độ sụt giảm xác suất khi chỉ giữ lại top-k cạnh quan trọng nhất.
  - **$\Delta	ext{Fidelity}$:** $	ext{Fidelity}^+ - 	ext{Fidelity}^-$.
  - **Sparsity:** Tỷ lệ nén cạnh của mặt nạ giải thích.
  - **Stability:** Độ ổn định của giải thích khi chạy lặp lại trên cùng một đồ thị.
  - **AUC Insertion / Deletion:** Diện tích dưới đường cong khi tuần tự thêm hoặc bớt các cạnh quan trọng.
- Thực thi **Cổng kiểm soát chất lượng Gate D17**: Kiểm định Paired Clustered Bootstrap 95% CI theo cụm trận đấu nhằm xác nhận xem explainer có vượt trội baseline ngẫu nhiên có ý nghĩa thống kê hay không.

## 2. Vị trí của bước trong pipeline
Đây là **Bước 8 — Đối sánh XAI & Kiểm định Gate D17 (`scripts/08_explain.py`, `scripts/09_evaluate_xai.py`, `src/xai_football/xai/`)**.

## 3. Input và nguồn dữ liệu
- Checkpoint MPNN tốt nhất: `models/official/mpnn_seed789_best.pt`
- Đồ thị PyG tập Test: `data/processed/official/graphs.pt` và `data/splits/official/test_matches.csv`
- Bảng kết quả XAI chính thức: `results/tables/official/xai_comparison.csv`, `sparsity_fidelity_curves.csv`, và manifest `xai_comparison_manifest.json`

## 4. Config/script/module sử dụng
- Cấu hình: `configs/xai.yaml`
- Scripts: `scripts/08_explain.py`, `scripts/09_evaluate_xai.py`
- Package XAI: `src/xai_football/xai/` (chứa `gnn_explainer.py`, `pg_explainer.py`, `metrics.py`)

## 5. Các bước xử lý & Đọc kết quả thực tế
Đoạn mã nạp bảng so sánh chất lượng giải thích của 4 thuật toán và kết quả kiểm định Gate D17:

In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display, Image

tables_dir = Path("../results/tables/official")
xai_comp_csv = tables_dir / "xai_comparison.csv"
manifest_p = tables_dir / "xai_comparison_manifest.json"

df_xai = pd.read_csv(xai_comp_csv)
print(f"Nạp thành công kết quả XAI từ: {xai_comp_csv}")
display(df_xai)

if manifest_p.exists():
    with open(manifest_p, encoding="utf-8") as f:
        manifest = json.load(f)
    print("\n=== MANIFEST THỰC NGHIỆM XAI GATE D17 ===")
    print(f"Số lượng đồ thị Test đánh giá: {manifest.get('n_graphs')}")
    print(f"Số cụm trận đấu (Clusters): {manifest.get('n_clusters')}")
    print("Gate D17 Results:")
    for k, v in manifest.get("gate_d17_results", {}).items():
        print(f"  - {k}: Status={v.get('status')} | Delta={v.get('delta_fidelity_vs_random'):+.4f} | 95% CI={v.get('cluster_ci_95')}")

## 6. Kết quả và cách diễn giải

### 6.1. Trực quan hóa đối sánh Fidelity và Đường cong Sparsity-Fidelity

In [ ]:
fig1 = Path("../results/figures/official/08_xai_fidelity_comparison.png")
fig2 = Path("../results/figures/official/08_xai_sparsity_fidelity_curves.png")

if fig1.exists():
    display(Image(filename=str(fig1), width=750))
if fig2.exists():
    display(Image(filename=str(fig2), width=750))

### 6.2. Diễn giải khoa học & Kết luận Gate D17
1. **Kiểm định Paired Clustered Bootstrap (Gate D17):**
   - Heuristic `last_edge`: $\Delta	ext{Fidelity} = +0.2779$, chênh lệch theo cặp so với Random là $+0.2282$, 95% Cluster CI $[+0.2077, +0.2502]$ hoàn toàn nằm trên 0 $ightarrow$ **PASS (Vượt qua Gate D17)**.
   - `gnn_explainer`: $\Delta	ext{Fidelity} = +0.1608$, chênh lệch theo cặp so với Random là $+0.1111$, 95% Cluster CI $[+0.0939, +0.1286]$ hoàn toàn nằm trên 0 $ightarrow$ **PASS (Vượt qua Gate D17)**.
   - `pg_explainer`: $\Delta	ext{Fidelity} = +0.0368$, thấp hơn baseline ngẫu nhiên Random ($+0.0496$), 95% Cluster CI $[-0.0306, +0.0049]$ vắt qua 0 $ightarrow$ **FAIL (Trượt Gate D17)**.
2. **Vai trò then chốt của đường chuyền cuối cùng:**
   - Cắt bỏ đường chuyền quyết định (`last_edge`) làm xác suất dự đoán cú sút giảm mạnh nhất ($	ext{Fidelity}^+ = 0.1237$), trong khi chỉ giữ lại đường chuyền này bảo toàn tối đa độ tự tin dự đoán ($	ext{Fidelity}^- = -0.1542$).
3. **Độ ổn định và tính thăm dò của PGExplainer:**
   - PGExplainer đạt độ ổn định gần như tuyệt đối (**0.9872** so với 0.2851 của GNNExplainer) cùng tốc độ suy luận nhanh gấp hàng trăm lần. Mặc dù trượt Gate D17 ở cấp đồ thị đơn lẻ, PGExplainer được lựa chọn cho quy trình Cross-Fitting toàn mùa mang tính chất **minh chứng phương pháp luận thăm dò (exploratory methodological proof-of-concept)**.

## 7. Kiểm tra chất lượng (Quality Checks)
Xác thực các tiêu chuẩn chất lượng của thực nghiệm XAI:

In [ ]:
checks = []
checks.append({
    "Tiêu chí": "Đủ 4 phương pháp giải thích XAI",
    "Kết quả": "PASS" if len(df_xai) == 4 else "FAIL"
})
checks.append({
    "Tiêu chí": "Đánh giá trên đủ 1.222 đồ thị Test (57 trận)",
    "Kết quả": "PASS" if (df_xai["n_graphs"] == 1222).all() else "FAIL"
})
checks.append({
    "Tiêu chí": "Fidelity+ và Fidelity- nằm trong đoạn [-1, 1]",
    "Kết quả": "PASS" if (df_xai["fidelity_plus"].between(-1, 1).all() and df_xai["fidelity_minus"].between(-1, 1).all()) else "FAIL"
})
checks.append({
    "Tiêu chí": "Last-edge và GNNExplainer vượt qua Gate D17 (CI > 0)",
    "Kết quả": "PASS" if (df_xai.loc[df_xai['explainer'].isin(['last_edge', 'gnn_explainer']), 'gate_d17_status'] == 'PASS').all() else "FAIL"
})

df_checks = pd.DataFrame(checks)
display(df_checks)
assert (df_checks["Kết quả"] == "PASS").all(), "Có kiểm tra chất lượng không đạt!"
print("Toàn bộ kiểm tra chất lượng XAI đều ĐẠT (PASS)!")

## 8. Output được lưu ở đâu
- Bảng số liệu: `results/tables/official/xai_comparison.csv`, `sparsity_fidelity_curves.csv`
- Manifest: `results/tables/official/xai_comparison_manifest.json`
- Biểu đồ: `results/figures/official/08_xai_fidelity_comparison.png/.svg`, `08_xai_sparsity_fidelity_curves.png/.svg`
- Chi tiết từng đồ thị: `results/xai/official/mpnn/official_test/`

## 9. Cách tái chạy
```bash
python scripts/08_explain.py --config configs/xai.yaml --model-path models/official/mpnn_seed789_best.pt --split test
python scripts/09_evaluate_xai.py --run-id official_test
```

## 10. Giới hạn và giả định
- Tập đồ thị giải thích XAI được lọc tối thiểu $\ge 4$ cạnh theo Quyết định D2 nhằm bảo đảm có đủ cấu trúc mạng để giải thích.
- Cổng Gate D17 sử dụng phân cụm theo trận đấu (`match_id`) để triệt tiêu tương quan nội tại giữa các tình huống trong cùng một trận.